# Trade Promotion & Demand Analytics — Exploratory Analysis

This notebook walks through the data integration, SQL feature layer, and key
findings behind the Streamlit dashboard in this repo. Run `src/generate_data.py`
and `src/sql_pipeline.py` first if `data/trade_promo.db` doesn't exist yet.


In [1]:
import sys
sys.path.append('../src')
import sqlite3
import pandas as pd
import numpy as np
import plotly.express as px

conn = sqlite3.connect('../data/trade_promo.db')
pd.set_option('display.max_columns', None)


## 1. Data integration & cleaning (SQL layer)

Fact table = retail scanner sell-out joined against store and SKU dimensions. See `src/sql_pipeline.py` for the full cleaning + view-building logic.

In [2]:
row_count = pd.read_sql("SELECT COUNT(*) AS n FROM fact_sales", conn)
print(f"Fact table rows: {row_count['n'].iloc[0]:,}")

pd.read_sql("""
    SELECT category, COUNT(DISTINCT sku_id) AS n_skus, SUM(revenue) AS total_revenue
    FROM fact_sales
    GROUP BY category ORDER BY total_revenue DESC
""", conn)


Fact table rows: 1,200,000


,category,n_skus,total_revenue
0,Personal Care,14,2.031633e+09
1,Snacks,11,1.573910e+09
2,Beverages,5,6.219089e+08
3,Home Care,6,6.196086e+08
4,Dairy,4,2.513984e+08


## 2. Promo uplift by category

Using the `vw_sku_promo_performance` view (promo vs. non-promo baseline average units).

In [3]:
perf = pd.read_sql("SELECT * FROM vw_sku_promo_performance", conn)
cat_uplift = perf.groupby("category")["uplift_pct"].mean().sort_values(ascending=False)
fig = px.bar(cat_uplift, orientation="h", title="Average promo uplift % by category")
fig.show()
cat_uplift


category
Dairy            68.325000
Home Care        60.200000
Personal Care    51.378571
Beverages        50.000000
Snacks           44.254545
Name: uplift_pct, dtype: float64

## 3. Price elasticity check

Does deeper discount consistently drive higher uplift, or does it plateau (diminishing returns)?

In [4]:
fig = px.scatter(perf, x="avg_discount_pct", y="uplift_pct", color="category",
                  trendline="ols", title="Discount depth vs. uplift %")
fig.show()


## 4. Model performance summary

See `models/metrics.json` for the held-out (chronological split) evaluation of the
XGBoost demand model, and `models/backtest_results.json` for the trade-spend
efficiency backtest (model-recommended discount depth vs. actual historical
discount depth on held-out promo events).

In [5]:
import json
metrics = json.load(open('../models/metrics.json'))
backtest = json.load(open('../models/backtest_results.json'))
print("Model metrics:", json.dumps(metrics, indent=2))
print("\nBacktest results:", json.dumps(backtest, indent=2))


Model metrics: {
  "test_mape_pct": 24.01,
  "test_rmse_units": 9.03,
  "test_mae_units": 5.28,
  "train_rows": 1020000,
  "test_rows": 180000,
  "split_date": "2024-05-24",
  "n_features": 25
}

Backtest results: {
  "n_promo_events_sampled": 400,
  "actual_net_margin_total": -73727.5,
  "recommended_net_margin_total": -15103.17,
  "actual_trade_spend_efficiency": -0.6144,
  "recommended_trade_spend_efficiency": -0.1259,
  "efficiency_lift_pct": 79.51
}


## 5. Key finding

Across a held-out sample of historical promo events, the discount depths that
were actually run frequently over-discounted relative to what the model
identifies as margin-optimal for that SKU/store/context — cutting into profit
faster than volume grew (margin dilution). Recommending the model's suggested
discount depth instead of the actual historical depth reduces realized
trade-spend losses substantially on this sample (see `efficiency_lift_pct`
above). This is a **backtested, held-out simulation**, not a live field
experiment — see the README for that caveat.